# B.L.U.S.H: ACT on dataset v5, batch 64, on a Colab A100 (K7 2026-11-22)

The v5 recipe (`sim/scripts/train_sim.sh act-teacher`) with four changes K7 asked for: **batch 64** (v5: 8),
**peak learning rate 3e-5** (v5: 1e-5; about the square-root rule for 8x the batch), warmup 2,000 steps and a floor of
3e-6, and full precision as v5 (bf16 was planned but saved no time on the A100: picture loading sets the pace). 100k steps, about 17 passes over the 384k frames
(v5: about 2). Everything else as v5: same dataset (`krish5831/blush_sim_teacher_v5`, so the picture defect and
dropout 0.1 of the post-mortem are still in), image augmentation on, torchcodec decoding, AdamW, weight decay 1e-4.

**Where things go**
- The dataset (7 GB) goes from the Hub straight to this machine's local disk (cell 4), never through Drive or the laptop.
- Every 5k steps a checkpoint goes to the private Hub repo `MODEL_REPO` (lerobot does this itself, and it is how the
  laptop's tests get the weights) and to Google Drive (`DRIVE_DIR`): the weights of every checkpoint, the full
  resumable state of 50k and of the newest one.
- The laptop pulls each checkpoint from the Hub and tests it in the simulator (`sim/scripts/colab_tests.sh`).

**Speed (smoke on an A100 40 GB, 12 CPU cores, 2026-11-22):** 0.42 s per step on the GPU + ~0.4 s waiting for
pictures (12 cores unpack ~82 samples/s) = ~0.8 s per step: 100k steps = **~22 h**, a checkpoint every ~66 min.
Asked for a G4 first; Colab had none for 30 min and fell back to a T4.

**Before you run:** Runtime -> Change runtime type -> **A100** (G4 if available). Secrets (key icon): `HF_TOKEN` with write access,
"Notebook access" on. Keep the tab open: a closed tab ends the session.

**After a disconnect:** run all again. Cell 7 finds the newest checkpoint on the Hub and resumes from it.

In [ ]:
# 1. The machine: GPU, CPU cores, RAM, local disk.
import os, shutil, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
CORES = os.cpu_count()
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"python {sys.version.split()[0]} | {CORES} CPU cores | {ram:.0f} GB RAM | "
      f"local disk free {shutil.disk_usage('/content').free / 1e9:.0f} GB")
WORK = "/content"

In [ ]:
# 2. lerobot, pinned to the SAME commit as the laptop (0.6.2, 5aa74557), so the checkpoints load in the local tests.
#    Then a real computation on the GPU: a torch build without this GPU's architecture still says "cuda available".
LEROBOT = "lerobot[training] @ git+https://github.com/huggingface/lerobot@5aa74557"
%pip install -q "{LEROBOT}"
TRAIN = shutil.which("lerobot-train") or "lerobot-train"
CHECK = ("import torch, torchcodec, lerobot; x = torch.randn(512, 512, device='cuda'); "
         "y = (x @ x).to(torch.bfloat16).float().sum().item(); "
         "print('lerobot', lerobot.__version__, 'torch', torch.__version__, 'torchcodec', torchcodec.__version__, "
         "'gpu', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0), 'matmul ok', y == y)")
r = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0 and "matmul ok True" in r.stdout, "torch can't compute on this GPU: read the lines above"

In [ ]:
# 3. Hugging Face login (token from Colab Secrets, never from the notebook) and the run's settings.
from google.colab import userdata
from huggingface_hub import whoami
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("logged in as", whoami()["name"])

RUN = "act_teacher_v5b64"
DATASET = "krish5831/blush_sim_teacher_v5"        # 800 teacher-v5 episodes (private), as v5
MODEL_REPO = "krish5831/blush_sim_act_v5b64"      # every checkpoint under checkpoints/<step>/ (private)
DRIVE_DIR = f"/content/drive/MyDrive/blush_training/{RUN}"
KEEP_STATE = {"050000"}                            # Drive keeps these steps' optimizer state too (K7's 50k stop)
STEPS, SAVE_FREQ = 100_000, 5_000
BATCH = 64
PEAK_LR, WARMUP, FLOOR_LR = 3e-5, 2_000, 3e-6
PRECISION = "no"      # K7 2026-11-22: full precision. bf16 saved only ~10 % of the GPU time on the A100 (0.38 vs 0.42 s)
                      # and none of the total, which picture loading sets (~0.4 s/step); the local tests run fp32
WORKERS = max(2, min(12, CORES))                   # every step decodes 128 AV1 frames + augmentation on the CPU
os.environ["HF_LEROBOT_HOME"] = f"{WORK}/lerobot_home"   # the dataset's local copy (cell 4)

COMMON = [
    f"--dataset.repo_id={DATASET}", "--dataset.image_transforms.enable=true", "--dataset.video_backend=torchcodec",
    "--policy.type=act", "--policy.device=cuda",
    "--use_policy_training_preset=false", "--optimizer.type=adamw", f"--optimizer.lr={PEAK_LR}",
    "--optimizer.weight_decay=1e-4", "--scheduler.type=cosine_decay_with_warmup", f"--scheduler.peak_lr={PEAK_LR}",
    f"--scheduler.decay_lr={FLOOR_LR}", f"--scheduler.num_warmup_steps={WARMUP}", f"--scheduler.num_decay_steps={STEPS}",
    f"--batch_size={BATCH}", f"--num_workers={WORKERS}", "--wandb.enable=false",
]
print("workers", WORKERS)


def run_live(cmd, log_path):
    """Runs lerobot-train and shows its log here as it comes, also into log_path. Stopping the cell stops training."""
    env = dict(os.environ, TQDM_MININTERVAL="30", PYTHONUNBUFFERED="1")
    lines = []
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        with open(log_path, "a") as f:
            for line in pr.stdout:
                if "INFO" in line and not line.startswith("INFO"):
                    line = line.replace("INFO", "\nINFO", 1)
                print(line, end="", flush=True); f.write(line); f.flush(); lines.append(line)
        pr.wait()
    except BaseException:
        pr.terminate()
        try:
            pr.wait(timeout=60)
        except subprocess.TimeoutExpired:
            pr.kill()
        print("\n-- stopped; the training process is ended too")
        raise
    if pr.returncode != 0:
        raise RuntimeError(f"lerobot-train stopped with an error (exit {pr.returncode}); log: {log_path}")
    return lines

In [ ]:
# 4. The dataset onto this machine's local disk (7 GB), then a picture check: two video files decoded here must give
#    exactly the pixels the laptop's torchcodec gives (fingerprints taken 2026-11-22). If they differ, the policy
#    would train on different pictures from v5's: stop and look, don't train.
import hashlib, time
from huggingface_hub import snapshot_download
from torchcodec.decoders import VideoDecoder
t = time.time()
ROOT = f"{os.environ['HF_LEROBOT_HOME']}/{DATASET}"
snapshot_download(DATASET, repo_type="dataset", local_dir=ROOT)
print(f"dataset at {ROOT} in {time.time() - t:.0f} s; local disk free {shutil.disk_usage('/content').free / 1e9:.0f} GB")
LAPTOP = {("videos/observation.images.base/chunk-000/file-031.mp4", 0): "c5564dcd30c8",
          ("videos/observation.images.base/chunk-000/file-031.mp4", 1): "35b8004835f9",
          ("videos/observation.images.base/chunk-000/file-031.mp4", 500): "908dc1c63138",
          ("videos/observation.images.base/chunk-000/file-031.mp4", 5000): "e1eee8e9f8dd",
          ("videos/observation.images.claw/chunk-000/file-010.mp4", 0): "19fe5a97ba04",
          ("videos/observation.images.claw/chunk-000/file-010.mp4", 1): "154622bb855e",
          ("videos/observation.images.claw/chunk-000/file-010.mp4", 500): "37fe391e48f8",
          ("videos/observation.images.claw/chunk-000/file-010.mp4", 5000): "f30f17bb654c"}
bad = []
for (f, i), want in LAPTOP.items():
    got = hashlib.md5(VideoDecoder(f"{ROOT}/{f}")[i].numpy().tobytes()).hexdigest()[:12]
    print(f, i, got, "ok" if got == want else f"DIFFERS (laptop {want})")
    bad += [] if got == want else [(f, i)]
assert not bad, "decoded pictures differ from the laptop's: don't train on them, tell K7"

In [ ]:
# 5. Google Drive, where every checkpoint is copied (cell 7). Colab asks once for permission.
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive folder:", DRIVE_DIR, "| Drive free:", f"{shutil.disk_usage('/content/drive').free / 1e9:.0f} GB",
      "(needs about 5 GB: 20 x 0.2 GB weights + 2 x 0.6 GB full state)")

In [ ]:
# 6. Smoke: 300 steps in bf16, then 100 in full precision. The loss must be a number, the pictures must load fast
#    enough (data_s well below updt_s), and bf16 must actually be faster. Nothing goes to the Hub or Drive here.
import re


def smoke(name, steps, precision):
    shutil.rmtree(f"{WORK}/{name}", ignore_errors=True)
    cmd = [TRAIN, *COMMON, f"--steps={steps}", f"--save_freq={steps}", "--log_freq=50",
           "--policy.push_to_hub=false", f"--output_dir={WORK}/{name}", f"--job_name={name}"]
    if precision != "no":                         # lerobot reads a literal "no" as False: pass the flag only for bf16
        cmd.append(f"--accelerator.mixed_precision={precision}")
    log = [l for l in run_live(cmd, f"{WORK}/{name}.log") if "updt_s:" in l]
    last = log[-1]
    loss, updt, data = (re.search(p, last).group(1) for p in (r"loss:([\d.na]+)", r"updt_s:([\d.]+)", r"data_s:([\d.]+)"))
    shutil.rmtree(f"{WORK}/{name}", ignore_errors=True)
    return loss, float(updt), float(data)

res = {"bf16": smoke("smoke_bf16", 300, "bf16"), "fp32": smoke("smoke_fp32", 100, "no")}
for k, (loss, updt, data) in res.items():
    print(f"{k}: loss {loss}, {updt:.3f} s per step on the GPU + {data:.3f} s loading -> "
          f"{STEPS} steps = {STEPS * (updt + data) / 3600:.1f} h, a checkpoint every {SAVE_FREQ * (updt + data) / 60:.0f} min")
assert res["bf16"][0] != "nan", "loss is nan in bf16: tell K7 (full precision is the fallback)"
if res["bf16"][2] > 0.5 * res["bf16"][1]:
    print("!! loading the pictures takes more than half the GPU time: the CPU is the limit")

In [ ]:
# 7. The run. Fresh if MODEL_REPO holds no checkpoint, else RESUMED from its newest one (run this cell again after a
#    disconnect). lerobot pushes each checkpoint to the Hub before it trains on; a thread here copies it to Drive
#    (weights of every step, the full state of 50k and of the newest) and keeps only the newest two on the local disk.
import datetime as dt, pathlib, threading, time
from huggingface_hub import HfApi

try:
    have = [f for f in HfApi().list_repo_files(MODEL_REPO) if f.startswith("checkpoints/")]
except Exception:
    have = []
stamp = dt.datetime.now().strftime("%m%d_%H%M%S")
OUT = f"{WORK}/train/{RUN}_{stamp}"
if have:
    print("resuming from", MODEL_REPO, "checkpoint", max(f.split("/")[1] for f in have))
    cmd = [TRAIN, "--resume=true", f"--config_path={MODEL_REPO}", f"--output_dir={OUT}"]
else:
    print("fresh run ->", MODEL_REPO)
    cmd = [TRAIN, *COMMON, f"--steps={STEPS}", f"--save_freq={SAVE_FREQ}", "--log_freq=100",
           "--save_checkpoint_to_hub=true", "--policy.push_to_hub=true", f"--policy.repo_id={MODEL_REPO}",
           "--policy.private=true", f"--output_dir={OUT}", f"--job_name={RUN}"]
    if PRECISION != "no":                         # lerobot reads a literal "no" as False: pass the flag only for bf16
        cmd.append(f"--accelerator.mixed_precision={PRECISION}")


def to_drive(out):
    """Every minute: copy finished checkpoints (lerobot points `last` at one once it is complete and on the Hub)."""
    done_drive = set()
    while True:
        time.sleep(60)
        try:
            ck = pathlib.Path(out, "checkpoints")
            last = ck / "last"
            if not last.is_symlink():
                continue
            newest = pathlib.Path(os.readlink(last)).name
            steps = sorted(p.name for p in ck.glob("[0-9]*") if p.is_dir() and p.name <= newest)
            for s in steps:
                if s in done_drive:
                    continue
                dst = pathlib.Path(DRIVE_DIR, "checkpoints", s)
                shutil.copytree(ck / s / "pretrained_model", dst / "pretrained_model", dirs_exist_ok=True)
                if s == newest or s in KEEP_STATE:
                    shutil.copytree(ck / s / "training_state", dst / "training_state", dirs_exist_ok=True)
                done_drive.add(s)
                print(f"\n[drive] checkpoint {s} copied to {dst}", flush=True)
            for p in pathlib.Path(DRIVE_DIR, "checkpoints").glob("[0-9]*/training_state"):
                if p.parent.name not in KEEP_STATE and p.parent.name != newest:
                    shutil.rmtree(p, ignore_errors=True)       # older full states: the weights stay
            for s in steps[:-2]:
                if s in done_drive:
                    shutil.rmtree(ck / s, ignore_errors=True)  # on the Hub and on Drive by now
        except Exception as e:                                  # Drive hiccup: try again next minute
            print(f"\n[drive] copy failed, retrying: {e}", flush=True)

threading.Thread(target=to_drive, args=(OUT,), daemon=True).start()
run_live(cmd, f"{WORK}/train_{RUN}_{stamp}.log")

In [ ]:
# 8. What is on the Hub and on Drive now.
from huggingface_hub import HfApi
print("Hub:", sorted({f.split("/")[1] for f in HfApi().list_repo_files(MODEL_REPO) if f.startswith("checkpoints/")}))
print("Drive:", sorted(os.listdir(f"{DRIVE_DIR}/checkpoints")) if os.path.isdir(f"{DRIVE_DIR}/checkpoints") else [])